# Bab 21: Penskalaan Multidimensi dan Visualisasi Dimensi Tinggi

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab21/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab21")
print("Folder kerja:", os.getcwd())

## Listing 21.1: MDS klasik: segitiga dan kota-kota Sumatera

`kode/bab21/mds_klasik.py`

In [ ]:
import numpy as np
import pandas as pd

def mds_klasik(D, k=2):
    n = len(D)
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * J @ (D ** 2) @ J
    nilai, vektor = np.linalg.eigh(B)
    urut = np.argsort(nilai)[::-1]
    nilai, vektor = nilai[urut], vektor[:, urut]
    return vektor[:, :k] * np.sqrt(np.maximum(nilai[:k], 0)), nilai

D = np.array([[0, 3, 4], [3, 0, 5], [4, 5, 0]], float)
X, nilai = mds_klasik(D)
print("segitiga 3-4-5, nilai eigen B:", nilai.round(3))
print("koordinat:\n", X.round(3))

KOTA = {"Banda Aceh": (5.55, 95.32), "Medan": (3.59, 98.67),
        "Pekanbaru": (0.51, 101.45), "Padang": (-0.95, 100.35),
        "Jambi": (-1.61, 103.61), "Bengkulu": (-3.80, 102.27),
        "Palembang": (-2.98, 104.75),
        "Bandar Lampung": (-5.43, 105.26)}
lat, lon = np.radians(list(KOTA.values())).T
dlat = lat[:, None] - lat[None]
dlon = lon[:, None] - lon[None]
h = np.sin(dlat / 2) ** 2 + np.cos(lat[:, None]) * np.cos(lat[None]) \
    * np.sin(dlon / 2) ** 2
D = 2 * 6371 * np.arcsin(np.sqrt(h))                             # km
print("\njarak udara (km), sebagian:")
print(pd.DataFrame(D, KOTA, KOTA).iloc[:4, :4].round(0).to_string())
X, nilai = mds_klasik(D)
print("nilai eigen terbesar:", (nilai[:4] / 1e3).round(1), "(ribu)")
peta = pd.DataFrame(X, KOTA, ["dim1", "dim2"]).round(0)
print(peta.to_string())
with open("gambar/bab21/kota.dat", "w") as f:
    f.write("x y nama\n")
    for k, r in peta.iterrows():
        f.write(f"{r.dim1:.0f} {r.dim2:.0f} {{{k}}}\n")

## Listing 21.2: Stress MDS metrik dan nonmetrik menurut dimensi

`kode/bab21/stress.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab21')
import sys; sys.path.insert(0, "kode/bab21")
import numpy as np
from scipy.spatial.distance import pdist, squareform
from sklearn.isotonic import IsotonicRegression
from sklearn.manifold import MDS
from siapkan import matriks_panel

D, nama = matriks_panel()
iu = np.triu_indices(len(D), 1)

def stress1(D, X):
    """Stress-1 Kruskal: jarak peta vs disparitas monoton."""
    d = pdist(X)
    dhat = IsotonicRegression().fit_transform(D[iu], d)
    return np.sqrt(((d - dhat) ** 2).sum() / (d ** 2).sum()), d, dhat

print("dim  stress metrik  stress nonmetrik")
hasil = []
for k in range(1, 6):
    s = {}
    for metrik in [True, False]:
        mds = MDS(k, metric_mds=metrik, metric="precomputed",
                  n_init=8, init="random", random_state=0,
                  normalized_stress=False)
        X = mds.fit_transform(D)
        s[metrik] = stress1(D, X)[0]
    hasil.append((k, s[True], s[False]))
    print(f"{k:3d}  {s[True]:13.3f}  {s[False]:16.3f}")
with open("gambar/bab21/stress.dat", "w") as f:
    f.write("k metrik nonmetrik\n")
    for k, a, b in hasil:
        f.write(f"{k} {a:.4f} {b:.4f}\n")

## Listing 21.3: PCA dan t-SNE untuk anggota Kopi Nusantara

`kode/bab21/tsne.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab21')
import sys; sys.path.insert(0, "kode/bab15")
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.preprocessing import StandardScaler
from siapkan import muat

df, X = muat()
Z = StandardScaler().fit_transform(X)
seg = KMeans(5, n_init=20, random_state=0).fit_predict(Z)
rng = np.random.default_rng(21)
i = rng.choice(len(Z), 2000, replace=False)
Z, seg = Z[i], seg[i]

peta = {"PCA": PCA(2).fit_transform(Z)}
for p in [5, 30, 100]:
    peta[f"t-SNE p={p}"] = TSNE(perplexity=p, init="pca",
                                random_state=0).fit_transform(Z)
print("metode        trustworthiness (k=10)")
for nama, Y in peta.items():
    print(f"{nama:13s} {trustworthiness(Z, Y, n_neighbors=10):.3f}")

with open("gambar/bab21/tsne.dat", "w") as f:
    f.write("px py tx ty seg\n")
    for a, b, s in zip(peta["PCA"], peta["t-SNE p=30"], seg):
        f.write(f"{a[0]:.3f} {a[1]:.3f} {b[0]:.2f} {b[1]:.2f} {s}\n")

## Listing 21.4: Studi kasus peta rasa Nusantara

`kode/bab21/studi_kasus.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab21')
import sys; sys.path.insert(0, "kode/bab21")
import numpy as np
import pandas as pd
from scipy.linalg import orthogonal_procrustes
from scipy.spatial import procrustes
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LinearRegression
from sklearn.manifold import MDS
from sklearn.preprocessing import StandardScaler
from siapkan import matriks_panel

D, nama = matriks_panel()
iu = np.triu_indices(len(D), 1)
def nmds(D, seed=0):
    m = MDS(2, metric_mds=False, metric="precomputed", n_init=16,
            init="random", random_state=seed, normalized_stress=False)
    return m.fit_transform(D)
X = nmds(D)
d = pdist(X)
dhat = IsotonicRegression().fit_transform(D[iu], d)
stress = np.sqrt(((d - dhat) ** 2).sum() / (d ** 2).sum())
rho = spearmanr(D[iu], d).statistic
print(f"MDS nonmetrik 2D: stress-1 {stress:.3f}")
print(f"Spearman(skor panel, jarak peta) {rho:.3f}")

rasa = pd.read_csv("kode/data/profil_rasa.csv").set_index("kopi")
rasa = rasa.loc[nama]
A = StandardScaler().fit_transform(rasa.iloc[:, 2:])
P = PCA(2).fit_transform(A)
_, _, beda = procrustes(X, P)
print(f"Procrustes peta panel vs peta PCA atribut: {1 - beda:.3f}")

R, _ = orthogonal_procrustes(X - X.mean(0), P - P.mean(0))
X = (X - X.mean(0)) @ R                  # putar agar sumbu 1 ~ PC1
ATR = ["tanah", "pahit", "keasaman", "bunga", "manis",
       "cokelat", "rasa_ikutan"]
Xs = X / X.std(0)
print("\npencocokan atribut ke peta (arah dan R^2):")
panah = {}
for a in ATR:
    lr = LinearRegression().fit(Xs, rasa[a])
    arah = lr.coef_ / np.linalg.norm(lr.coef_)
    panah[a] = arah * np.sqrt(lr.score(Xs, rasa[a]))
    print(f"  {a:11s} arah ({arah[0]:+.2f}, {arah[1]:+.2f})"
          f"  R^2 {lr.score(Xs, rasa[a]):.2f}")

print("\nkesesuaian setiap panelis dengan peta bersama:")
for p in ["made", "ayu", "rizal", "tiur", "yosef"]:
    Dp, _ = matriks_panel(f"skor_{p}")
    _, _, b = procrustes(X, nmds(Dp))
    r = spearmanr(Dp[iu], D[iu]).statistic
    print(f"  {p:6s} Spearman dgn rata-rata {r:.2f}, "
          f"Procrustes {1 - b:.3f}")

stabil = []
for s in range(1, 6):
    _, _, b = procrustes(X, nmds(D, seed=s))
    stabil.append(1 - b)
print(f"\nkestabilan antar-inisialisasi: {min(stabil):.3f}"
      f" - {max(stabil):.3f}")

with open("gambar/bab21/peta.dat", "w") as f:
    f.write("x y pulau nama\n")
    kode = {p: i for i, p in enumerate(rasa.pulau.unique())}
    for k, (x, y) in zip(nama, Xs):
        f.write(f"{x:.3f} {y:.3f} {kode[rasa.pulau[k]]} {{{k}}}\n")
with open("gambar/bab21/panah.dat", "w") as f:
    for a, v in panah.items():
        f.write(f"{v[0]:.3f} {v[1]:.3f} {{{a.replace('_', ' ')}}}\n")
with open("gambar/bab21/shepard.dat", "w") as f:
    f.write("skor jarak disparitas\n")
    o = np.argsort(D[iu])
    for i in o:
        f.write(f"{D[iu][i]:.2f} {d[i]:.3f} {dhat[i]:.3f}\n")
print("urutan pulau:", list(kode))